# EDA CIC-IDS2017

In [33]:
import os
os.chdir(
    '/Users/a.zhukov/dev/phd/notebooks'
)

In [34]:
import warnings
from pathlib import Path
from zipfile import ZipFile

import cufflinks as cf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

cf.go_offline()
cf.set_config_file(offline=True, world_readable=False, theme="white")

warnings.filterwarnings("ignore", category=FutureWarning, module="cufflinks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

## Оформление графиков

Все графики оформляет одна функция `style`, чтобы они выглядели одинаково.

In [3]:
# Цвета типов трафика. На всех графиках один тип всегда одного цвета.
TRAFFIC_COLORS = {
    "Норма": "#2a78d6",
    "Атака": "#eb6834",
    "Попытка атаки": "#1baf7a",
}

# Шкала для тепловых карт, где значения бывают и меньше, и больше нуля:
# синий для отрицательных, серый для нуля, красный для положительных.
BLUE_GRAY_RED = [[0.0, "#2a78d6"], [0.5, "#f0efec"], [1.0, "#e34948"]]

BACKGROUND_COLOR = "#fcfcfb"
GRID_COLOR = "#e1e0d9"
AXIS_COLOR = "#c3c2b7"
TEXT_COLOR = "#0b0b0b"


def style(fig, title, x_title=None, y_title=None, height=450):
    """Задаёт графику общий вид: заголовок, подписи осей, шрифт, фон и сетку."""
    top_margin, bottom_margin = 110, 60
    plot_height = height - top_margin - bottom_margin

    fig.update_layout(
        # Заголовок стоит в левом верхнем углу, на 28 пикселей ниже края
        title=dict(
            text=title,
            font=dict(size=17, color=TEXT_COLOR),
            x=0,
            xref="paper",
            y=1 - 28 / height,
            yref="container",
            yanchor="top",
        ),
        # Легенда стоит между заголовком и областью графика, на 26 пикселей выше области
        legend=dict(orientation="h", x=0, y=1 + 26 / plot_height, yanchor="bottom", bgcolor="rgba(0,0,0,0)"),
        height=height,
        margin=dict(l=70, r=30, t=top_margin, b=bottom_margin),
        font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", size=13, color=TEXT_COLOR),
        paper_bgcolor=BACKGROUND_COLOR,
        plot_bgcolor=BACKGROUND_COLOR,
        separators=", ",  # дробная часть через запятую, разряды через пробел
        bargap=0.35,
        barcornerradius=4,
    )
    # exponentformat="none": числа на осях пишутся полностью, без английских сокращений k и M
    axis_style = dict(
        gridcolor=GRID_COLOR, linecolor=AXIS_COLOR, zeroline=False, automargin=True, exponentformat="none"
    )
    fig.update_xaxes(**axis_style)
    fig.update_yaxes(**axis_style)
    if x_title:
        fig.update_xaxes(title_text=x_title)
    if y_title:
        fig.update_yaxes(title_text=y_title)
    # cufflinks обводит столбцы рамкой цвета столбца. Заменяем её тонкой рамкой цвета фона:
    # она отделяет друг от друга части составного столбца.
    fig.update_traces(marker_line=dict(color=BACKGROUND_COLOR, width=1), selector=dict(type="bar"))
    return fig


def with_spaces(number):
    """Записывает целое число с пробелами между разрядами: 2099976 -> '2 099 976'."""
    return f"{number:,}".replace(",", " ")

## 1. Загрузка данных и словарь русских названий

In [ ]:
DATA_ARCHIVE = Path("../data/CICIDS2017_improved.zip")

DAY_FILES = {
    "monday.csv": "Понедельник",
    "tuesday.csv": "Вторник",
    "wednesday.csv": "Среда",
    "thursday.csv": "Четверг",
    "friday.csv": "Пятница",
}


def load_dataset(archive_path):
    days = []
    with ZipFile(archive_path) as archive:
        for file_name, day_name in DAY_FILES.items():
            with archive.open(file_name) as csv_file:
                day = pd.read_csv(csv_file, parse_dates=["Timestamp"])
            day["Day"] = day_name
            days.append(day)
    dataset = pd.concat(days, ignore_index=True)
    dataset["Day"] = pd.Categorical(dataset["Day"], categories=list(DAY_FILES.values()), ordered=True)
    return dataset


df = load_dataset(DATA_ARCHIVE)
print(f"Потоков: {with_spaces(len(df))}, колонок: {df.shape[1]}")

Потоков: 2 099 976, колонок: 92


In [7]:
df.head()

,id,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd RST Flags,Bwd RST Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWR Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Fwd Segment Size Avg,Bwd Segment Size Avg,Fwd Bytes/Bulk Avg,Fwd Packet/Bulk Avg,Fwd Bulk Rate Avg,Bwd Bytes/Bulk Avg,Bwd Packet/Bulk Avg,Bwd Bulk Rate Avg,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,FWD Init Win Bytes,Bwd Init Win Bytes,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Label,Attempted Category,Day
0,1,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,2017-07-03 11:56:34.157427,119719148,231,0,0,0,0,0,0.000000,0.000000,0,0,0.0,0.0,0.000000,1.929516,5.205180e+05,2.274526e+06,20694308,0,119719148,5.205180e+05,2.274526e+06,20694308,0,0,0.000000e+00,0.000000e+00,0,0,0,0,0,0,0,0,0,0,1.929516,0.000000,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0.0,0.000000,0.000000,0.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,8721243.6,8.761761e+06,22509459,17,12685486.0,5.296658e+06,20694308,6499982,-1,-1,0,BENIGN,-1,Понедельник
1,2,192.168.10.9-192.168.10.3-123-123-17,192.168.10.9,123,192.168.10.3,123,17,2017-07-03 11:56:55.428911,65511209,6,6,288,288,48,48,48.000000,0.000000,48,48,48.0,0.0,8.792388,0.183175,5.955564e+06,1.925808e+07,64004884,3,65511097,1.310222e+07,2.846297e+07,64004961,3,65511105,1.310222e+07,2.846299e+07,64004995,3,0,0,0,0,0,0,48,48,0.091587,0.091587,48,48,48.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,1.0,48.000000,48.000000,48.0,0,0,0,0,0,0,0,24,0,24,0,0,5,8,1506210.0,0.000000e+00,1506210,1506210,64004884.0,0.000000e+00,64004884,64004884,-1,-1,0,BENIGN,-1,Понедельник
2,3,192.168.10.12-224.0.0.251-5353-5353-17,192.168.10.12,5353,224.0.0.251,5353,17,2017-07-03 11:57:21.057686,113976922,267,0,20447,0,153,37,76.580524,44.140625,0,0,0.0,0.0,179.395966,2.342579,4.284847e+05,3.701805e+06,48523116,0,113976922,4.284847e+05,3.701805e+06,48523116,0,0,0.000000e+00,0.000000e+00,0,0,0,0,0,0,0,0,2136,0,2.342579,0.000000,37,153,76.580524,44.140625,1948.394807,0,0,0,0,0,0,0,0,0.0,76.580524,76.580524,0.0,2044,26,4794,0,0,0,1,76,0,0,0,0,266,8,2745981.5,5.491934e+06,10983883,14,25498178.0,1.883305e+07,48523116,5463561,-1,-1,0,BENIGN,-1,Понедельник
3,4,192.168.10.12-152.2.133.52-123-123-17,192.168.10.12,123,152.2.133.52,123,17,2017-07-03 11:57:31.568196,67037196,8,8,384,384,48,48,48.000000,0.000000,48,48,48.0,0.0,11.456326,0.238673,4.469146e+06,1.427166e+07,55956316,44916,66990997,9.570142e+06,2.047764e+07,56001261,992873,66991090,9.570156e+06,2.047819e+07,56002515,991683,0,0,0,0,0,0,64,64,0.119337,0.119337,48,48,48.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,1.0,48.000000,48.000000,48.0,0,0,0,0,0,0,0,24,0,24,0,0,7,8,11034681.0,0.000000e+00,11034681,11034681,55956316.0,0.000000e+00,55956316,55956316,-1,-1,0,BENIGN,-1,Понедельник
4,5,192.168.10.12-45.79.11.217-123-123-17,192.168.10.12,123,45.79.11.217,123,17,2017-07-03 11:57:30.571719,68045057,8,8,384,384,48,48,48.000000,0.000000,48,48,48.0,0.0,11.286639,0.235138,4.536337e+06,1.452505e+07,56943904,57451,67987500,9.712500e+06,2.085582e+07,57001355,989295,67987529,9.712504e+06,2.085587e+07,57001461,989348,0,0,0,0,0,0,64,64,0.117569,0.11

### Дополнительные колонки

`Attempted` = попытка атаки, создаем отдельный класс

2 новых колонки:
- `Attack`: вид атаки без слова `Attempted`;
- `Traffic Type`: одно из трёх значений: "Норма", "Атака", "Попытка атаки"

In [11]:
ATTEMPTED_SUFFIX = " - Attempted"

is_benign = df["Label"] == "BENIGN"
is_attempted = df["Label"].str.endswith(ATTEMPTED_SUFFIX)

df["Attack"] = df["Label"].str.replace(ATTEMPTED_SUFFIX, "", regex=False)
df["Traffic Type"] = np.select([is_benign, is_attempted], ["Норма", "Попытка атаки"], default="Атака")

df["Traffic Type"].value_counts()

Traffic Type
Норма            1582566
Атака             505431
Попытка атаки      11979
Name: count, dtype: int64

### Словарь русских названий

Сделано как в документе

In [12]:
ATTRIBUTE_GROUPS = {
    "1. Идентификатор потока": {
        "Flow ID": "Идентификатор потока",
    },
    "2. Источник": {
        "Src IP": "IP-адрес источника",
        "Src Port": "Порт источника",
    },
    "3. Назначение": {
        "Dst IP": "IP-адрес назначения",
        "Dst Port": "Порт назначения",
    },
    "4. Протокол": {
        "Protocol": "Протокол",
    },
    "5. Время начала потока": {
        "Timestamp": "Время начала потока",
    },
    "6. Длительность потока": {
        "Flow Duration": "Длительность потока, мкс",
    },
    "7. Число пакетов": {
        "Total Fwd Packet": "Пакетов вперёд",
        "Total Bwd packets": "Пакетов обратно",
    },
    "8. Объём переданных данных": {
        "Total Length of Fwd Packet": "Байт вперёд",
        "Total Length of Bwd Packet": "Байт обратно",
    },
    "9. Размер пакета по направлениям": {
        "Fwd Packet Length Max": "Наибольший размер пакета вперёд, байт",
        "Fwd Packet Length Min": "Наименьший размер пакета вперёд, байт",
        "Fwd Packet Length Mean": "Средний размер пакета вперёд, байт",
        "Fwd Packet Length Std": "Ст. отклонение размера пакета вперёд",
        "Bwd Packet Length Max": "Наибольший размер пакета обратно, байт",
        "Bwd Packet Length Min": "Наименьший размер пакета обратно, байт",
        "Bwd Packet Length Mean": "Средний размер пакета обратно, байт",
        "Bwd Packet Length Std": "Ст. отклонение размера пакета обратно",
    },
    "10. Скорость потока": {
        "Flow Bytes/s": "Байт в секунду",
        "Flow Packets/s": "Пакетов в секунду",
    },
    "11. Интервалы между пакетами": {
        "Flow IAT Mean": "Средний интервал между пакетами, мкс",
        "Flow IAT Std": "Ст. отклонение интервала между пакетами",
        "Flow IAT Max": "Наибольший интервал между пакетами, мкс",
        "Flow IAT Min": "Наименьший интервал между пакетами, мкс",
    },
    "12. Интервалы между пакетами по направлениям": {
        "Fwd IAT Total": "Сумма интервалов вперёд, мкс",
        "Fwd IAT Mean": "Средний интервал вперёд, мкс",
        "Fwd IAT Std": "Ст. отклонение интервала вперёд",
        "Fwd IAT Max": "Наибольший интервал вперёд, мкс",
        "Fwd IAT Min": "Наименьший интервал вперёд, мкс",
        "Bwd IAT Total": "Сумма интервалов обратно, мкс",
        "Bwd IAT Mean": "Средний интервал обратно, мкс",
        "Bwd IAT Std": "Ст. отклонение интервала обратно",
        "Bwd IAT Max": "Наибольший интервал обратно, мкс",
        "Bwd IAT Min": "Наименьший интервал обратно, мкс",
    },
    "13. Флаг PSH по направлениям": {
        "Fwd PSH Flags": "Пакетов с флагом PSH вперёд",
        "Bwd PSH Flags": "Пакетов с флагом PSH обратно",
    },
    "14. Флаг URG по направлениям": {
        "Fwd URG Flags": "Пакетов с флагом URG вперёд",
        "Bwd URG Flags": "Пакетов с флагом URG обратно",
    },
    "15. Длина заголовков": {
        "Fwd Header Length": "Длина заголовков вперёд, байт",
        "Bwd Header Length": "Длина заголовков обратно, байт",
    },
    "16. Скорость по направлениям": {
        "Fwd Packets/s": "Пакетов в секунду вперёд",
        "Bwd Packets/s": "Пакетов в секунду обратно",
    },
    "17. Размер пакета без деления на направления": {
        "Packet Length Min": "Наименьший размер пакета, байт",
        "Packet Length Max": "Наибольший размер пакета, байт",
        "Packet Length Mean": "Средний размер пакета, байт",
        "Packet Length Std": "Ст. отклонение размера пакета",
        "Packet Length Variance": "Дисперсия размера пакета",
    },
    "18. Число пакетов с флагами TCP": {
        "FIN Flag Count": "Пакетов с флагом FIN",
        "SYN Flag Count": "Пакетов с флагом SYN",
        "RST Flag Count": "Пакетов с флагом RST",
        "PSH Flag Count": "Пакетов с флагом PSH",
        "ACK Flag Count": "Пакетов с флагом ACK",
        "URG Flag Count": "Пакетов с флагом URG",
    },
    "19. Флаги CWR и ECE": {
        "CWR Flag Count": "Пакетов с флагом CWR",
        "ECE Flag Count": "Пакетов с флагом ECE",
    },
    "20. Отношение обратных пакетов к прямым": {
        "Down/Up Ratio": "Отношение числа пакетов обратно к числу пакетов вперёд",
    },
    "21. Средний размер пакета": {
        "Average Packet Size": "Средний размер пакета в потоке, байт",
    },
    "22. Средний размер сегмента": {
        "Fwd Segment Size Avg": "Средний размер сегмента вперёд, байт",
        "Bwd Segment Size Avg": "Средний размер сегмента обратно, байт",
    },
    "23. Передача блоками": {
        "Fwd Bytes/Bulk Avg": "Байт в блоке вперёд, в среднем",
        "Fwd Packet/Bulk Avg": "Пакетов в блоке вперёд, в среднем",
        "Fwd Bulk Rate Avg": "Скорость передачи блоков вперёд, в среднем",
        "Bwd Bytes/Bulk Avg": "Байт в блоке обратно, в среднем",
        "Bwd Packet/Bulk Avg": "Пакетов в блоке обратно, в среднем",
        "Bwd Bulk Rate Avg": "Скорость передачи блоков обратно, в среднем",
    },
    "24. Подпотоки": {
        "Subflow Fwd Packets": "Пакетов в подпотоке вперёд",
        "Subflow Fwd Bytes": "Байт в подпотоке вперёд",
        "Subflow Bwd Packets": "Пакетов в подпотоке обратно",
        "Subflow Bwd Bytes": "Байт в подпотоке обратно",
    },
    "25. Начальное окно TCP вперёд": {
        "FWD Init Win Bytes": "Начальное окно TCP вперёд, байт",
    },
    "26. Начальное окно TCP обратно": {
        "Bwd Init Win Bytes": "Начальное окно TCP обратно, байт",
    },
    "27. Пакеты с данными": {
        "Fwd Act Data Pkts": "Пакетов с данными вперёд",
    },
    "28. Наименьший размер сегмента": {
        "Fwd Seg Size Min": "Наименьший размер сегмента вперёд, байт",
    },
    "29. Время активности": {
        "Active Mean": "Среднее время активности, мкс",
        "Active Std": "Ст. отклонение времени активности",
        "Active Max": "Наибольшее время активности, мкс",
        "Active Min": "Наименьшее время активности, мкс",
    },
    "30. Время простоя": {
        "Idle Mean": "Среднее время простоя, мкс",
        "Idle Std": "Ст. отклонение времени простоя",
        "Idle Max": "Наибольшее время простоя, мкс",
        "Idle Min": "Наименьшее время простоя, мкс",
    },
    "31. Метка класса": {
        "Label": "Метка",
    },
    "Нет в документе, есть в исправленной версии датасета": {
        "id": "Номер строки в файле дня",
        "Fwd RST Flags": "Пакетов с флагом RST вперёд",
        "Bwd RST Flags": "Пакетов с флагом RST обратно",
        "ICMP Code": "Код ICMP",
        "ICMP Type": "Тип ICMP",
        "Total TCP Flow Time": "Общее время TCP-соединения, мкс",
        "Attempted Category": "Категория попытки атаки",
    },
    "Добавлено в этом блокноте": {
        "Day": "День недели",
        "Attack": "Вид атаки",
        "Traffic Type": "Тип трафика",
    },
}

# Словарь "колонка -> русское название". Используется во всех подписях.
RU = {
    column: russian_name
    for columns in ATTRIBUTE_GROUPS.values()
    for column, russian_name in columns.items()
}

# То же самое в виде таблицы: видно, к какому пункту документа относится колонка.
attributes = pd.DataFrame(
    [
        (column, russian_name, group)
        for group, columns in ATTRIBUTE_GROUPS.items()
        for column, russian_name in columns.items()
    ],
    columns=["Колонка", "Русское название", "Пункт документа"],
).set_index("Колонка")

# Проверяем, что русское название есть у каждой колонки и что лишних названий нет
columns_without_name = set(df.columns) - set(RU)
names_without_column = set(RU) - set(df.columns)
assert not columns_without_name, f"Нет русского названия: {columns_without_name}"
assert not names_without_column, f"Таких колонок нет в данных: {names_without_column}"

attributes

,Русское название,Пункт документа
Колонка,,
Flow ID,Идентификатор потока,1. Идентификатор потока
Src IP,IP-адрес источника,2. Источник
Src Port,Порт источника,2. Источник
Dst IP,IP-адрес назначения,3. Назначение
Dst Port,Порт назначения,3. Назначение
Protocol,Протокол,4. Протокол
Timestamp,Время начала потока,5. Время начала потока
Flow Duration,"Длительность потока, мкс",6. Длительность потока
Total Fwd Packet,Пакетов вперёд,7. Число пакетов


In [13]:
def label_ru(label):
    """Метка для подписей: 'BENIGN' -> 'Норма', 'DoS Hulk - Attempted' -> 'DoS Hulk (попытка)'.

    Сами названия атак не переводим: это названия программ и приёмов.
    """
    if label == "BENIGN":
        return "Норма"
    return label.replace(ATTEMPTED_SUFFIX, " (попытка)")


# Русские подписи для всех меток датасета
LABEL_RU = {label: label_ru(label) for label in df["Label"].unique()}


def show_table(table, digits=2):
    """Показывает таблицу с русскими названиями признаков и меток и с пробелами между разрядами."""
    translation = {**RU, **LABEL_RU}
    table = table.rename(index=translation, columns=translation)
    display(table.style.format(precision=digits, thousands=" ", decimal=",", na_rep=""))

## 2. Обзор данных

In [14]:
memory_gb = df.memory_usage(deep=True).sum() / 1e9

print("Потоков:", with_spaces(len(df)))
print("Первый поток:", df["Timestamp"].min().strftime("%d.%m.%Y %H:%M:%S"))
print("Последний поток:", df["Timestamp"].max().strftime("%d.%m.%Y %H:%M:%S"))
print(f"Память: {memory_gb:.1f} ГБ")

Потоков: 2 099 976
Первый поток: 03.07.2017 11:55:58
Последний поток: 07.07.2017 20:02:40
Память: 2.4 ГБ


In [ ]:
show_table(df.head(3).T)

,0,1,2
Номер строки в файле дня,1,2,3
Идентификатор потока,8.6.0.1-8.0.6.4-0-0-0,192.168.10.9-192.168.10.3-123-123-17,192.168.10.12-224.0.0.251-5353-5353-17
IP-адрес источника,8.6.0.1,192.168.10.9,192.168.10.12
Порт источника,0,123,5 353
IP-адрес назначения,8.0.6.4,192.168.10.3,224.0.0.251
Порт назначения,0,123,5 353
Протокол,0,17,17
Время начала потока,2017-07-03 11:56:34.157427,2017-07-03 11:56:55.428911,2017-07-03 11:57:21.057686
"Длительность потока, мкс",119 719 148,65 511 209,113 976 922
Пакетов вперёд,231,6,267


### Профайлинг

запускать не нужно, результат уже в report.html

In [20]:
from ydata_profiling import ProfileReport

In [ ]:
profile = ProfileReport(numeric)

In [32]:
profile.to_file('report.html')

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
DATA_TYPE_NAMES = {
    "int64": "целое число",
    "float64": "дробное число",
    "object": "текст",
    "datetime64[ns]": "дата и время",
    "category": "категория",
}

numeric = df.select_dtypes("number")

passport = attributes.copy()
passport["Тип данных"] = df.dtypes.astype(str).map(DATA_TYPE_NAMES)
passport["Разных значений"] = df.nunique()
passport["Доля нулей, %"] = (numeric == 0).mean() * 100
passport["Минимум"] = numeric.min()
passport["Медиана"] = numeric.median()
passport["Максимум"] = numeric.max()


In [ ]:
show_table(passport.reset_index())

## 3. Качество данных

Проверяем пропуски, бесконечные и отрицательные значения, повторы строк и признаки, которые почти не меняются.

In [10]:
problems = pd.DataFrame(
    {
        "Пропусков": df.isna().sum(),
        "Бесконечных значений": np.isinf(numeric).sum(),
        "Отрицательных значений": (numeric < 0).sum(),
    }
)
problems = problems.fillna(0).astype(int)

# Оставляем только колонки, где нашлось хотя бы одно такое значение
problems = problems[problems.sum(axis=1) > 0]
show_table(problems)

,Пропусков,Бесконечных значений,Отрицательных значений
Категория попытки атаки,0,0,2 087 997
"Длина заголовков обратно, байт",0,0,472
Байт в секунду,0,5,0
Пакетов в секунду,0,5,0
"Длина заголовков вперёд, байт",0,0,309
Код ICMP,0,0,2 099 373
Тип ICMP,0,0,2 099 373


Разбираем каждую находку.

In [11]:
# Коды ICMP. Значение -1 стоит у потоков, которые не относятся к протоколу ICMP (номер протокола 1).
not_icmp = df["Protocol"] != 1
print("Потоков не ICMP:", with_spaces(not_icmp.sum()))
print("Из них с кодом ICMP, равным -1:", with_spaces((df.loc[not_icmp, "ICMP Code"] == -1).sum()))

# Категория попытки атаки. Значение -1 стоит у потоков без слова Attempted в метке.
not_attempted = df["Traffic Type"] != "Попытка атаки"
print("Потоков без пометки Attempted:", with_spaces(not_attempted.sum()))
print("Из них с категорией -1:", with_spaces((df.loc[not_attempted, "Attempted Category"] == -1).sum()))

Потоков не ICMP: 2 099 373
Из них с кодом ICMP, равным -1: 2 099 373
Потоков без пометки Attempted: 2 087 997
Из них с категорией -1: 2 087 997


In [12]:
# Бесконечная скорость. Скорость равна объёму, делённому на длительность.
# У потока с нулевой длительностью получается деление на ноль.
RATE_COLUMNS = ["Flow Bytes/s", "Flow Packets/s"]

infinite_rate = np.isinf(df["Flow Packets/s"])
print("Потоков с бесконечной скоростью:", infinite_rate.sum())
print("Из них с нулевой длительностью:", (df.loc[infinite_rate, "Flow Duration"] == 0).sum())

# Заменяем бесконечность на пропуск, чтобы она не искажала средние значения и графики
df[RATE_COLUMNS] = df[RATE_COLUMNS].replace(np.inf, np.nan)

Потоков с бесконечной скоростью: 5
Из них с нулевой длительностью: 5


In [ ]:
HEADER_COLUMNS = ["Fwd Header Length", "Bwd Header Length"]

show_table(df[HEADER_COLUMNS].agg(["min", "max"]).rename(index={"min": "Минимум", "max": "Максимум"}))

negative_header = (df[HEADER_COLUMNS] < 0).any(axis=1)
print("Потоков с отрицательной длиной заголовков:", negative_header.sum())
show_table(
    df[negative_header]
    .groupby("Label")
    .size()
    .rename("Потоков с отрицательной длиной заголовков")
    .to_frame()
)

,"Длина заголовков вперёд, байт","Длина заголовков обратно, байт"
Минимум,-32 708,-32 764
Максимум,32 748,32 680


Потоков с отрицательной длиной заголовков: 593


,Потоков с отрицательной длиной заголовков
Label,
Норма,585
Heartbleed,3
Infiltration,5


In [ ]:
# Дубликаты
duplicates = df.drop(columns="id").duplicated().sum()
print("Дубликатов:", duplicates)

Дубликатов: 0


In [15]:
# Признаки, которые почти всегда равны нулю
mostly_zero = passport[passport["Доля нулей, %"] >= 99]
show_table(mostly_zero[["Русское название", "Доля нулей, %", "Максимум"]].reset_index(), digits=3)

,Колонка,Русское название,"Доля нулей, %",Максимум
0,Fwd URG Flags,Пакетов с флагом URG вперёд,"99,996","16,000"
1,Bwd URG Flags,Пакетов с флагом URG обратно,"100,000","2,000"
2,URG Flag Count,Пакетов с флагом URG,"99,996","16,000"
3,CWR Flag Count,Пакетов с флагом CWR,"99,964","23,000"
4,ECE Flag Count,Пакетов с флагом ECE,"99,967","14,000"
5,Subflow Bwd Packets,Пакетов в подпотоке обратно,"99,998","1,000"


**Что видно.**
- Пропусков нет, полных повторов строк нет.
- В колонках `ICMP Code`, `ICMP Type` и `Attempted Category` значение −1 означает «не относится к этому потоку». Это условное обозначение, а не ошибка.
- Бесконечная скорость стоит у 5 потоков с нулевой длительностью. В расчётах ниже она заменена пропуском.
- Отрицательная длина заголовков стоит у 593 потоков. Все значения лежат между −32 768 и 32 767, так бывает при переполнении 16-битного счётчика. Перед обучением модели такие значения нужно исправить или убрать.
- 6 признаков равны нулю больше чем в 99% потоков: это флаги URG, CWR, ECE и число пакетов в подпотоке обратно.

## 4. Метки классов

In [16]:
type_counts = df["Traffic Type"].value_counts().rename("Потоков").to_frame()
type_counts["Доля, %"] = type_counts["Потоков"] / len(df) * 100
show_table(type_counts)

,Потоков,"Доля, %"
Traffic Type,,
Норма,1 582 566,"75,36"
Атака,505 431,"24,07"
Попытка атаки,11 979,"0,57"


In [17]:
label_counts = (
    df.groupby(["Label", "Traffic Type"])
    .size()
    .rename("Потоков")
    .reset_index()
    .sort_values("Потоков", ascending=False)
)
label_counts["Доля, %"] = label_counts["Потоков"] / len(df) * 100

show_table(label_counts.set_index("Label"), digits=4)

,Тип трафика,Потоков,"Доля, %"
Label,,,
Норма,Норма,1 582 566,"75,3611"
Portscan,Атака,159 066,"7,5747"
DoS Hulk,Атака,158 468,"7,5462"
DDoS,Атака,95 144,"4,5307"
Infiltration - Portscan,Атака,71 767,"3,4175"
DoS GoldenEye,Атака,7 567,"0,3603"
Botnet (попытка),Попытка атаки,4 067,"0,1937"
FTP-Patator,Атака,3 972,"0,1891"
DoS Slowloris,Атака,3 859,"0,1838"


In [ ]:
chart_data = label_counts.pivot(index="Label", columns="Traffic Type", values="Потоков")
chart_data = chart_data.loc[label_counts["Label"][::-1], list(TRAFFIC_COLORS)]
chart_data.index = chart_data.index.map(LABEL_RU)

fig = chart_data.iplot(kind="barh", barmode="stack", colors=TRAFFIC_COLORS, opacity=1, asFigure=True)
style(
    fig,
    title="Число потоков по меткам<br><sup>Шкала логарифмическая: каждое деление в 10 раз больше предыдущего</sup>",
    x_title="Потоков",
    height=820,
)
fig.update_xaxes(type="log", dtick=1)  # dtick=1: отметки только на 10, 100, 1000 и так далее
fig.show()

### Какие атаки шли в какой день

In [19]:
day_counts = pd.crosstab(df["Day"], df["Traffic Type"])[list(TRAFFIC_COLORS)]

fig = day_counts.iplot(kind="bar", barmode="stack", colors=TRAFFIC_COLORS, opacity=1, asFigure=True)
style(fig, title="Число потоков по дням недели", y_title="Потоков").show()

In [20]:
attack_days = pd.crosstab(df.loc[~is_benign, "Label"], df.loc[~is_benign, "Day"])
show_table(attack_days)

Day,Вторник,Среда,Четверг,Пятница
Label,,,,
Botnet,0,0,0,736
Botnet (попытка),0,0,0,4 067
DDoS,0,0,0,95 144
DoS GoldenEye,0,7 567,0,0
DoS GoldenEye (попытка),0,80,0,0
DoS Hulk,0,158 468,0,0
DoS Hulk (попытка),0,581,0,0
DoS Slowhttptest,0,1 740,0,0
DoS Slowhttptest (попытка),0,3 368,0,0


### Почему поток помечен как попытка атаки

https://intrusion-detection.distrinet-research.be/CNS2022/CICIDS2017.html

In [21]:
ATTEMPTED_CATEGORIES = {
    0: "0. Атакующий не отправил данных",
    1: "1. Порт или система закрыты",
    2: "2. Поток запуска или остановки атаки",
    3: "3. Нет вредоносного содержимого",
    4: "4. Побочный поток атаки",
    5: "5. Атака выполнена с ошибкой",
    6: "6. Цель перестала отвечать",
}

attempted = df[is_attempted]
attempted_reasons = pd.crosstab(attempted["Label"], attempted["Attempted Category"].map(ATTEMPTED_CATEGORIES))
attempted_reasons.columns.name = "Причина"
show_table(attempted_reasons)

Причина,0. Атакующий не отправил данных,1. Порт или система закрыты,2. Поток запуска или остановки атаки,3. Нет вредоносного содержимого,4. Побочный поток атаки,5. Атака выполнена с ошибкой,6. Цель перестала отвечать
Label,,,,,,,
Botnet (попытка),0,4 067,0,0,0,0,0
DoS GoldenEye (попытка),80,0,0,0,0,0,0
DoS Hulk (попытка),579,0,0,0,2,0,0
DoS Slowhttptest (попытка),563,0,0,0,1,0,2 804
DoS Slowloris (попытка),1 705,0,3,0,1,138,0
FTP-Patator (попытка),10,0,2,0,0,0,0
Infiltration (попытка),42,0,3,0,0,0,0
SSH-Patator (попытка),0,0,0,27,0,0,0
Web Attack - Brute Force (попытка),1 214,0,7,0,71,0,0


**Что видно:**
- Нормальный трафик занимает 75,4% потоков, атаки 24,1%, попытки атак 0,6%.
- Классы очень неравные. У Portscan и DoS Hulk около 159 тысяч потоков, а у 5 видов атак меньше 100: Web Attack - Brute Force (73), Infiltration (36), Web Attack - XSS (18), Web Attack - SQL Injection (13), Heartbleed (11).
- В понедельник атак нет. Каждый вид атаки записан только в один день.
- У четырёх видов атак попыток больше, чем самих атак: Botnet (4 067 против 736), DoS Slowhttptest (3 368 против 1 740), Web Attack - Brute Force (1 292 против 73), Web Attack - XSS (655 против 18).
- Самая частая причина пометки «попытка»: атакующий не отправил данных. У Botnet причина другая: порт или система были закрыты.

## 5. Трафик во времени

Запись каждый день шла примерно с 11:45 до 20:10.

In [27]:
# Число потоков каждого типа за каждые 10 минут
flows_per_10_minutes = (
    df.groupby([pd.Grouper(key="Timestamp", freq="10min"), "Traffic Type"])
    .size()
    .unstack(fill_value=0)
    .asfreq("10min", fill_value=0)  # добавляем ночные интервалы с нулём потоков
)[list(TRAFFIC_COLORS)]

fig = flows_per_10_minutes.iplot(kind="scatter", colors=TRAFFIC_COLORS, width=2, asFigure=True)
style(
    fig,
    title="Число потоков за 10 минут",
    x_title="Время начала потока, UTC",
    y_title="Потоков за 10 минут",
    height=480,
)
# Отметки на оси времени: 12:00, 15:00 и 18:00 каждого дня. Дату пишем один раз в день.
first_day, last_day = df["Timestamp"].min().normalize(), df["Timestamp"].max().normalize()
tick_times, tick_labels = [], []
for day in pd.date_range(first_day, last_day):
    for hour in (12, 15, 18):
        tick_times.append(day + pd.Timedelta(hours=hour))
        tick_labels.append(f"{hour}:00<br>{day:%d.%m}" if hour == 12 else f"{hour}:00")

# Прячем с оси время с 20:30 до 11:30, когда записи нет
fig.update_xaxes(
    rangebreaks=[dict(bounds=[20.5, 11.5], pattern="hour")],
    tickvals=tick_times,
    ticktext=tick_labels,
)
fig.update_layout(hovermode="x unified")
fig.show()

### Расписание атак

Время первого и последнего потока каждой атаки. Попытки атак не учитываются.

In [23]:
attacks = df[df["Traffic Type"] == "Атака"]

schedule = attacks.groupby("Label").agg(
    **{
        "День": ("Day", "first"),
        "Начало": ("Timestamp", "min"),
        "Конец": ("Timestamp", "max"),
        "Потоков": ("Label", "size"),
    }
)
schedule["От начала до конца, мин"] = (schedule["Конец"] - schedule["Начало"]).dt.total_seconds() / 60
schedule = schedule.sort_values("Начало")
schedule["Начало"] = schedule["Начало"].dt.strftime("%H:%M:%S")
schedule["Конец"] = schedule["Конец"].dt.strftime("%H:%M:%S")

show_table(schedule, digits=1)

,День,Начало,Конец,Потоков,"От начала до конца, мин"
Label,,,,,
FTP-Patator,Вторник,12:19:20,13:20:13,3 972,"60,9"
SSH-Patator,Вторник,17:09:01,19:04:15,2 961,"115,2"
DoS Slowloris,Среда,12:48:54,13:10:11,3 859,"21,3"
DoS Slowhttptest,Среда,13:15:37,13:35:18,1 740,"19,7"
DoS Hulk,Среда,13:43:23,14:01:34,158 468,"18,2"
DoS GoldenEye,Среда,14:10:03,14:19:49,7 567,"9,8"
Heartbleed,Среда,18:12:15,18:32:18,11,"20,0"
Web Attack - Brute Force,Четверг,12:19:27,12:59:41,73,"40,2"
Web Attack - XSS,Четверг,13:16:16,13:34:28,18,"18,2"


**Что видно.**
- Три атаки дают всплески, которые во много раз выше обычного уровня трафика: DoS Hulk в среду, Portscan и DDoS в пятницу.
- Все потоки одной атаки укладываются в промежуток от 7 минут до 2 часов. Если делить данные на обучающую и проверочную части по дням, некоторые виды атак попадут только в одну из частей.

## 6. Протоколы, порты и адреса

In [24]:
PROTOCOL_NAMES = {0: "0 (протокол не определён)", 1: "1 (ICMP)", 6: "6 (TCP)", 17: "17 (UDP)"}

protocol_counts = pd.crosstab(df["Protocol"].map(PROTOCOL_NAMES), df["Traffic Type"])[list(TRAFFIC_COLORS)]
protocol_counts.index.name = RU["Protocol"]
show_table(protocol_counts)

Traffic Type,Норма,Атака,Попытка атаки
Протокол,,,
0 (протокол не определён),1 508,2,0
1 (ICMP),526,77,0
17 (UDP),998 855,237,0
6 (TCP),581 677,505 115,11 979


In [25]:
# Те же числа в процентах: какая доля потоков каждого типа идёт по каждому протоколу
show_table(protocol_counts / protocol_counts.sum() * 100)

Traffic Type,Норма,Атака,Попытка атаки
Протокол,,,
0 (протокол не определён),"0,10","0,00","0,00"
1 (ICMP),"0,03","0,02","0,00"
17 (UDP),"63,12","0,05","0,00"
6 (TCP),"36,76","99,94","100,00"


**Что видно:** Атаки почти целиком идут по TCP, а нормальный трафик на 63% состоит из потоков UDP. Это важно для сравнения признаков: у потоков UDP нет флагов и окна TCP, поэтому разница между нормой и атаками может объясняться протоколом, а не атакой. В разделах 7 и 8 нормальный трафик TCP и UDP показан отдельно.

In [26]:
# Названия служб для самых частых портов. У остальных портов показываем только номер.
PORT_SERVICES = {
    21: "FTP",
    22: "SSH",
    53: "DNS",
    80: "HTTP",
    88: "Kerberos",
    123: "NTP",
    135: "RPC",
    137: "NetBIOS",
    139: "NetBIOS",
    389: "LDAP",
    443: "HTTPS",
    445: "SMB",
    3268: "LDAP",
    8080: "HTTP",
}


def port_label(port):
    """Подпись порта: '80 (HTTP)' для известных портов, '49666' для остальных."""
    if port in PORT_SERVICES:
        return f"{port} ({PORT_SERVICES[port]})"
    return str(port)


def plot_top_ports(traffic_type, count=10):
    """Рисует порты назначения, на которые пришло больше всего потоков заданного типа."""
    ports = df.loc[df["Traffic Type"] == traffic_type, "Dst Port"]
    top_ports = ports.value_counts().head(count)
    share_of_top = top_ports.sum() / len(ports) * 100

    # Переворачиваем порядок: на горизонтальном графике первая строка рисуется внизу
    chart_data = top_ports[::-1].rename("Потоков").to_frame()
    chart_data.index = chart_data.index.map(port_label)

    fig = chart_data.iplot(kind="barh", colors=TRAFFIC_COLORS[traffic_type], opacity=1, asFigure=True)
    style(
        fig,
        title=(
            f"{RU['Dst Port']}: {count} самых частых, тип трафика «{traffic_type}»"
            f"<br><sup>На эти порты приходится {share_of_top:.0f}% потоков этого типа</sup>"
        ),
        x_title="Потоков",
        height=430,
    )
    fig.update_yaxes(type="category")
    fig.update_layout(showlegend=False)
    fig.show()


plot_top_ports("Норма")
plot_top_ports("Атака")

### Кто кого атакует

Пары адресов для каждого вида атаки. Показаны пары, на которые приходится не меньше 10 потоков.

In [27]:
attack_pairs = (
    attacks.groupby(["Attack", "Src IP", "Dst IP"])
    .size()
    .rename("Потоков")
    .reset_index()
    .sort_values(["Attack", "Потоков"], ascending=[True, False])
)
show_table(attack_pairs[attack_pairs["Потоков"] >= 10].set_index("Attack"))

,IP-адрес источника,IP-адрес назначения,Потоков
Attack,,,
Botnet,192.168.10.15,205.174.165.73,220
Botnet,192.168.10.9,205.174.165.73,151
Botnet,192.168.10.14,205.174.165.73,139
Botnet,192.168.10.8,205.174.165.73,117
Botnet,192.168.10.5,205.174.165.73,109
DDoS,172.16.0.1,192.168.10.50,95 144
DoS GoldenEye,172.16.0.1,192.168.10.50,7 567
DoS Hulk,172.16.0.1,192.168.10.50,158 468
DoS Slowhttptest,172.16.0.1,192.168.10.50,1 740


**Что видно.**
- Нормальный трафик сосредоточен на нескольких портах, больше всего потоков идёт на порт 53 (DNS).
- Больше половины потоков атак приходит на порт 80
- Большинство атак идёт с одного адреса `172.16.0.1` на один адрес `192.168.10.50`. Можно переобучиться

## 7. Числовые признаки по пунктам документа

ось Х `log10(1 + значение)`

In [28]:
HISTOGRAM_BINS = 40

is_tcp = df["Protocol"] == 6
is_udp = df["Protocol"] == 17

# Группы потоков для сравнения: название группы и отбор её строк
COMPARED_GROUPS = {
    "Норма, TCP": is_benign & is_tcp,
    "Норма, UDP": is_benign & is_udp,
    "Атака": df["Traffic Type"] == "Атака",
}

# Норма нарисована двумя оттенками синего, атака оранжевым, как на остальных графиках
GROUP_COLORS = {
    "Норма, TCP": "#184f95",
    "Норма, UDP": "#5598e7",
    "Атака": "#eb6834",
}

NUMBER_UNITS = [(1e9, " млрд"), (1e6, " млн"), (1e3, " тыс."), (1, "")]


def compress(values):
    """Сжимает шкалу: log10(1 + значение). Ноль остаётся нулём, 9 превращается в 1, 999 999 в 6."""
    return np.log10(1 + values)


def short_number(value):
    """Короткая запись числа для подписей: 1000 -> '1 тыс.', 2500000 -> '2,5 млн'."""
    for size, unit in NUMBER_UNITS:
        if value >= size:
            return f"{value / size:.3g}".replace(".", ",") + unit
    # Сюда попадают числа меньше единицы
    return f"{value:.3g}".replace(".", ",")


def axis_ticks(max_value):
    """Отметки для сжатой оси: 0, 1, 10, 100 и так далее.

    Возвращает положения отметок на сжатой оси и подписи в исходных единицах.
    """
    largest_power = int(np.log10(max_value + 1))
    values = [0] + [10**power for power in range(largest_power + 1)]
    # На длинной оси оставляем отметки через одну, иначе подписи сливаются
    if len(values) > 6:
        values = values[::2]
    return [compress(value) for value in values], [short_number(value) for value in values]


def describe_columns(columns):
    """Сводка по признакам: доля нулей, границы значений по всем потокам и медианы по группам."""
    values = df[columns]
    summary = pd.DataFrame(
        {
            "Доля нулей, %": (values == 0).mean() * 100,
            "Минимум": values.min(),
            "Медиана": values.median(),
            "Среднее": values.mean(),
            "99-й процентиль": values.quantile(0.99),
            "Максимум": values.max(),
        }
    )
    for group_name, group_rows in COMPARED_GROUPS.items():
        summary[f"Медиана: {group_name}"] = values[group_rows].median()
    return summary


def plot_distributions(columns, title):
    """Рисует для каждого признака долю потоков по значениям, отдельно для каждой группы."""
    columns_in_row = min(3, len(columns))
    rows = int(np.ceil(len(columns) / columns_in_row))
    fig = make_subplots(
        rows=rows,
        cols=columns_in_row,
        subplot_titles=[RU[column] for column in columns],
        horizontal_spacing=0.08,
        vertical_spacing=0.28 / rows,
    )

    for number, column in enumerate(columns):
        row, col = number // columns_in_row + 1, number % columns_in_row + 1

        # Отрицательные значения и пропуски на сжатой оси показать нельзя, их отбрасываем.
        # Сколько таких значений, видно в разделе о качестве данных.
        can_plot = df[column] >= 0
        axis_end = compress(df.loc[can_plot, column].max())

        # Делим сжатую ось на равные интервалы
        bin_edges = np.linspace(0, axis_end, HISTOGRAM_BINS + 1)

        # Границы интервалов в исходных единицах для всплывающей подсказки
        original_edges = 10**bin_edges - 1
        hints = [
            f"от {short_number(left)} до {short_number(right)}"
            for left, right in zip(original_edges[:-1], original_edges[1:])
        ]

        for group_name, group_rows in COMPARED_GROUPS.items():
            values = df.loc[can_plot & group_rows, column]
            flows_in_bin, _ = np.histogram(compress(values), bins=bin_edges)
            share = flows_in_bin / len(values) * 100
            fig.add_trace(
                go.Scatter(
                    # Ступенчатая линия: доля держится от левой границы интервала до правой.
                    # Последнюю точку повторяем, чтобы дорисовать последний интервал до конца.
                    x=bin_edges,
                    y=np.append(share, share[-1]),
                    text=hints + hints[-1:],
                    name=group_name,
                    legendgroup=group_name,
                    showlegend=(number == 0),
                    mode="lines",
                    line=dict(color=GROUP_COLORS[group_name], width=2, shape="hv"),
                    hovertemplate="%{text}<br>%{y:.1f}% потоков",
                ),
                row=row,
                col=col,
            )

        tick_positions, tick_labels = axis_ticks(10**axis_end - 1)
        fig.update_xaxes(
            tickvals=tick_positions,
            ticktext=tick_labels,
            range=[-0.03 * axis_end, 1.03 * axis_end],  # небольшой отступ, чтобы были видны ноль и максимум
            row=row,
            col=col,
        )
        fig.update_yaxes(ticksuffix="%", rangemode="tozero", row=row, col=col)

    style(
        fig,
        title=f"{title}<br><sup>По горизонтали значение признака, по вертикали доля потоков группы с таким значением</sup>",
        height=170 + 250 * rows,
    )
    fig.update_annotations(font_size=13)  # названия признаков над маленькими графиками
    fig.show()


def show_group(group):
    """Показывает таблицу и графики для одного пункта документа."""
    columns = list(ATTRIBUTE_GROUPS[group])
    show_table(describe_columns(columns))
    plot_distributions(columns, title=group)

### Пункт 6. Длительность сессии
Время от первого до последнего пакета потока в микросекундах.

In [29]:
show_group("6. Длительность потока")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Длительность потока, мкс","0,00",0,"61 826,00","12 440 277,46","118 104 102,00",119 999 998,"5 958 021,00","30 658,00","37 999,00"


### Пункт 7. Число пакетов
Сколько пакетов прошло вперёд (от источника) и обратно (от назначения).

In [30]:
show_group("7. Число пакетов")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Пакетов вперёд,"0,00",0,"2,00","12,52","60,00",219 759,"12,00","2,00","7,00"
Пакетов обратно,"2,62",0,"2,00","14,21","75,00",291 922,"10,00","2,00","4,00"


### Пункт 8. Объём переданных данных
Суммарный размер пакетов в байтах вперёд и обратно.

In [31]:
show_group("8. Объём переданных данных")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Байт вперёд,"14,59",0,"80,00","570,42","7 018,00",12 870 252,"717,00","70,00","20,00"
Байт обратно,"15,81",0,"218,00","21 919,85","101 535,00",655 452 323,"3 597,00","174,00","11 595,00"


### Пункт 9. Размер пакета по направлениям
Наибольший, наименьший и средний размер пакета и стандартное отклонение размера, отдельно вперёд и обратно. Большое отклонение означает, что в потоке были пакеты очень разного размера.

In [32]:
show_group("9. Размер пакета по направлениям")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Наибольший размер пакета вперёд, байт","14,59","0,00","45,00","210,17","2 162,00","24 820,00","425,00","42,00","20,00"
"Наименьший размер пакета вперёд, байт","52,18","0,00","0,00","21,23","78,00","1 472,00","0,00","42,00","0,00"
"Средний размер пакета вперёд, байт","14,59","0,00","40,67","49,78","322,60","5 775,50","52,00","42,00","2,50"
Ст. отклонение размера пакета вперёд,"60,34","0,00","0,00","60,66","674,67","7 116,54","118,51","0,00","7,07"
"Наибольший размер пакета обратно, байт","15,81","0,00","128,00","1 231,97","10 136,00","19 530,00","1 430,00","103,00","4 344,00"
"Наименьший размер пакета обратно, байт","52,77","0,00","0,00","53,46","268,00","1 983,00","0,00","102,00","0,00"
"Средний размер пакета обратно, байт","15,81","0,00","114,00","410,30","2 898,75","4 370,69","283,22","102,00","1 449,38"
Ст. отклонение размера пакета обратно,"60,86","0,00","0,00","469,53","4 414,37","6 715,74","527,52","0,00","1 563,03"


### Пункт 10. Скорость потока
Сколько байт и пакетов передавалось за секунду.

In [33]:
show_group("10. Скорость потока")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Байт в секунду,"14,39","0,00","3 864,20","466 026,90","2 663 265,31","253 000 000,00","294,84","8 359,07","940,19"
Пакетов в секунду,"0,00","0,03","73,69","19 318,49","500 000,00","3 000 000,00","2,09","126,04","363,62"


### Пункт 11. Интервалы между пакетами
Время между соседними пакетами потока в микросекундах: среднее, стандартное отклонение, наибольшее и наименьшее.

In [34]:
show_group("11. Интервалы между пакетами")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Средний интервал между пакетами, мкс","0,00","0,00","18 959,00","920 510,85","22 042 061,58","69 000 029,00","519 427,62","10 472,33","2 997,08"
Ст. отклонение интервала между пакетами,"23,46","0,00","16 301,77","1 708 723,19","37 499 080,84","84 819 214,86","1 687 661,51","98,73","5 103,36"
"Наибольший интервал между пакетами, мкс","0,00","0,00","50 628,00","4 921 717,87","71 898 898,75","119 999 735,00","5 582 036,00","30 548,00","15 011,00"
"Наименьший интервал между пакетами, мкс","3,87","0,00","3,00","108 770,31","267 465,25","69 000 029,00","3,00","3,00","3,00"


### Пункт 12. Интервалы между пакетами по направлениям
Те же интервалы, но отдельно между пакетами вперёд и отдельно между пакетами обратно. Добавлена сумма интервалов.

In [35]:
show_group("12. Интервалы между пакетами по направлениям")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Сумма интервалов вперёд, мкс","22,18","0,00","48,00","12 370 611,25","118 095 419,50","119 999 998,00","5 928 709,00","3,00","37 016,00"
"Средний интервал вперёд, мкс","22,18","0,00","48,00","2 034 974,00","64 066 599,00","119 999 570,00","1 000 086,50","3,00","5 718,00"
Ст. отклонение интервала вперёд,"55,63","0,00","0,00","1 374 343,20","21 621 375,58","83 228 041,46","2 226 166,11","0,00","8 141,00"
"Наибольший интервал вперёд, мкс","22,18","0,00","48,00","4 890 524,07","71 873 637,00","119 999 869,00","5 567 886,00","3,00","22 951,00"
"Наименьший интервал вперёд, мкс","24,03","0,00","3,00","1 217 605,89","64 066 599,00","119 999 570,00","48,00","3,00","1,00"
"Сумма интервалов обратно, мкс","24,63","0,00","48,00","11 626 410,01","117 966 343,50","119 999 992,00","5 802 796,00","3,00","15 674,00"
"Средний интервал обратно, мкс","24,63","0,00","48,00","2 075 826,64","61 572 042,75","119 976 977,00","1 145 159,20","3,00","2 739,00"
Ст. отклонение интервала обратно,"60,51","0,00","0,00","1 228 664,66","22 938 140,54","84 620 454,92","2 205 664,06","0,00","4 475,13"
"Наибольший интервал обратно, мкс","24,63","0,00","48,00","4 383 701,95","71 158 736,00","119 991 381,00","5 563 456,00","3,00","10 519,00"
"Наименьший интервал обратно, мкс","25,17","0,00","3,00","1 258 139,24","61 572 042,75","119 976 977,00","4,00","3,00","3,00"


### Пункт 13. Флаг PSH по направлениям
Сколько пакетов с флагом PSH прошло вперёд и обратно. Флаг PSH просит получателя сразу передать данные приложению. У потоков UDP значение всегда 0.

In [36]:
show_group("13. Флаг PSH по направлениям")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Пакетов с флагом PSH вперёд,"62,39",0,"0,00","1,40","16,00",5 520,"3,00","0,00","1,00"
Пакетов с флагом PSH обратно,"62,93",0,"0,00","1,76","22,00",9 761,"3,00","0,00","1,00"


### Пункт 14. Флаг URG по направлениям
Сколько пакетов с флагом срочности URG прошло вперёд и обратно.

In [37]:
show_group("14. Флаг URG по направлениям")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Пакетов с флагом URG вперёд,"100,00",0,"0,00","0,00","0,00",16,"0,00","0,00","0,00"
Пакетов с флагом URG обратно,"100,00",0,"0,00","0,00","0,00",2,"0,00","0,00","0,00"


### Пункт 15. Длина заголовков
Суммарная длина заголовков пакетов в байтах вперёд и обратно. Отрицательные значения на графики не попадают, о них сказано в разделе о качестве данных.

In [38]:
show_group("15. Длина заголовков")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Длина заголовков вперёд, байт","0,10",-32 708,"32,00","177,68","1 576,00",32 748,"312,00","16,00","152,00"
"Длина заголовков обратно, байт","2,63",-32 764,"20,00","177,88","1 960,00",32 680,"264,00","16,00","104,00"


### Пункт 16. Скорость по направлениям
Число пакетов в секунду отдельно вперёд и обратно.

In [39]:
show_group("16. Скорость по направлениям")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Пакетов в секунду вперёд,"0,00","0,00","38,80","14 007,69","500 000,00","2 000 000,00","1,15","64,03","195,26"
Пакетов в секунду обратно,"2,62","0,00","32,60","5 310,75","55 555,56","2 000 000,00","0,94","59,22","45,79"


### Пункт 17. Размер пакета без деления на направления
Наименьший, наибольший и средний размер пакета, стандартное отклонение и дисперсия размера по всем пакетам потока.

In [40]:
show_group("17. Размер пакета без деления на направления")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Наименьший размер пакета, байт","52,32","0,00","0,00","20,87","78,00","1 408,00","0,00","42,00","0,00"
"Наибольший размер пакета, байт","14,39","0,00","130,00","1 269,72","10 136,00","24 820,00","1 448,00","104,00","4 344,00"
"Средний размер пакета, байт","14,39","0,00","81,00","212,02","1 055,91","2 125,67","188,69","72,00","702,71"
Ст. отклонение размера пакета,"16,93","0,00","51,96","378,78","2 798,91","4 693,16","422,95","35,80","1 315,70"
Дисперсия размера пакета,"16,93","0,00","2 700,00","581 904,36","7 833 875,19","22 025 757,21","178 888,10","1 281,33","1 731 071,10"


### Пункт 18. Число пакетов с флагами TCP
Сколько пакетов потока несли каждый из флагов FIN, SYN, RST, PSH, ACK, URG.

In [41]:
show_group("18. Число пакетов с флагами TCP")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Пакетов с флагом FIN,"63,34",0,"0,00","0,68","3,00",60,"2,00","0,00","1,00"
Пакетов с флагом SYN,"50,75",0,"0,00","0,95","4,00",52,"2,00","0,00","2,00"
Пакетов с флагом RST,"70,92",0,"0,00","0,42","3,00",190,"0,00","0,00","1,00"
Пакетов с флагом PSH,"62,19",0,"0,00","3,16","36,00",9 861,"7,00","0,00","2,00"
Пакетов с флагом ACK,"49,46",0,"1,00","24,08","131,00",511 681,"20,00","0,00","10,00"
Пакетов с флагом URG,"100,00",0,"0,00","0,00","0,00",16,"0,00","0,00","0,00"


### Пункт 19. Флаги CWR и ECE
Сколько пакетов несли флаги CWR и ECE. Этими флагами узлы сообщают друг другу о перегрузке сети.

In [42]:
show_group("19. Флаги CWR и ECE")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Пакетов с флагом CWR,"99,96",0,"0,00","0,00","0,00",23,"0,00","0,00","0,00"
Пакетов с флагом ECE,"99,97",0,"0,00","0,00","0,00",14,"0,00","0,00","0,00"


### Пункт 20. Отношение обратных пакетов к прямым
Число пакетов обратно, делённое на число пакетов вперёд. Значение 0,5 означает, что на каждые два отправленных пакета пришёл один ответный.

In [43]:
show_group("20. Отношение обратных пакетов к прямым")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Отношение числа пакетов обратно к числу пакетов вперёд,"2,62","0,00","1,00","0,91","1,55","16,93","0,89","1,00","0,88"


### Пункт 21. Средний размер пакета
Средний размер пакета в потоке.

In [44]:
show_group("21. Средний размер пакета")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Средний размер пакета в потоке, байт","14,39","0,00","81,00","212,02","1 055,91","2 125,67","188,69","72,00","702,71"


### Пункт 22. Средний размер сегмента

In [45]:
show_group("22. Средний размер сегмента")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Средний размер сегмента вперёд, байт","14,59","0,00","40,67","49,78","322,60","5 775,50","52,00","42,00","2,50"
"Средний размер сегмента обратно, байт","15,81","0,00","114,00","410,30","2 898,75","4 370,69","283,22","102,00","1 449,38"


### Пункт 23. Передача блоками

In [46]:
show_group("23. Передача блоками")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Байт в блоке вперёд, в среднем","95,86",0,"0,00","72,72","939,00",1 827 185,"0,00","0,00","0,00"
"Пакетов в блоке вперёд, в среднем","95,86",0,"0,00","0,24","5,00",3 812,"0,00","0,00","0,00"
"Скорость передачи блоков вперёд, в среднем","95,86",0,"0,00","184 773,40","57 057,25",974 000 000,"0,00","0,00","0,00"
"Байт в блоке обратно, в среднем","86,11",0,"0,00","4 062,81","50 699,00",156 761 876,"0,00","0,00","0,00"
"Пакетов в блоке обратно, в среднем","86,11",0,"0,00","2,04","29,00",58 270,"0,00","0,00","0,00"
"Скорость передачи блоков обратно, в среднем","86,11",0,"0,00","2 501 866,88","45 116 731,00",1 369 000 000,"0,00","0,00","0,00"


### Пункт 24. Подпотоки
Число пакетов и байт в подпотоках вперёд и обратно.

In [47]:
show_group("24. Подпотоки")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Пакетов в подпотоке вперёд,"97,38",0,"0,00","0,03","1,00",1,"0,00","0,00","0,00"
Байт в подпотоке вперёд,"14,85",0,"21,00","25,73","174,00",1 777,"27,00","21,00","1,00"
Пакетов в подпотоке обратно,"100,00",0,"0,00","0,00","0,00",1,"0,00","0,00","0,00"
Байт в подпотоке обратно,"15,88",0,"56,00","185,72","1 052,00",2 090,"134,00","51,00","682,00"


### Пункты 25 и 26. Начальное окно TCP
Размер окна TCP в первом пакете каждого направления: сколько байт сторона готова принять. У потоков UDP значение равно 0.

In [48]:
show_group("25. Начальное окно TCP вперёд")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Начальное окно TCP вперёд, байт","48,14",0,"503,00","10 036,33","65 535,00",65 535,"8 192,00","0,00","29 200,00"


In [49]:
show_group("26. Начальное окно TCP обратно")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Начальное окно TCP обратно, байт","61,01",0,"0,00","753,90","30 016,00",65 535,"245,00","0,00","229,00"


### Пункт 27. Пакеты с данными
Сколько пакетов вперёд несли данные, а не только служебную информацию.

In [50]:
show_group("27. Пакеты с данными")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Пакетов с данными вперёд,"27,75",0,"1,00","2,22","19,00",9 130,"4,00","1,00","1,00"


### Пункт 28. Наименьший размер сегмента

In [51]:
show_group("28. Наименьший размер сегмента")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Наименьший размер сегмента вперёд, байт","0,45",0,"20,00","17,05","40,00",44,"20,00","8,00","24,00"


### Пункт 29. Время активности

In [52]:
show_group("29. Время активности")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Среднее время активности, мкс","78,08","0,00","0,00","126 698,15","3 001 469,25","110 097 488,00","37 117,00","0,00","0,00"
Ст. отклонение времени активности,"89,75","0,00","0,00","59 818,65","1 537 445,01","74 154 397,74","0,00","0,00","0,00"
"Наибольшее время активности, мкс","78,08","0,00","0,00","227 506,22","5 235 834,50","110 097 488,00","64 842,00","0,00","0,00"
"Наименьшее время активности, мкс","78,08","0,00","0,00","92 405,68","1 877 904,25","110 097 488,00","23 241,00","0,00","0,00"


### Пункт 30. Время простоя

In [53]:
show_group("30. Время простоя")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
"Среднее время простоя, мкс","77,76","0,00","0,00","4 433 516,63","68 884 002,25","119 999 735,00","5 579 089,00","0,00","0,00"
Ст. отклонение времени простоя,"89,51","0,00","0,00","290 453,37","8 156 620,46","76 914 831,60","0,00","0,00","0,00"
"Наибольшее время простоя, мкс","77,76","0,00","0,00","4 677 768,59","71 898 898,75","119 999 735,00","5 582 036,00","0,00","0,00"
"Наименьшее время простоя, мкс","77,76","0,00","0,00","4 145 343,91","68 884 002,25","119 999 735,00","5 316 453,00","0,00","0,00"


### Доп признаки вне документа

In [54]:
ADDED_COLUMNS = ["Fwd RST Flags", "Bwd RST Flags", "Total TCP Flow Time"]

show_table(describe_columns(ADDED_COLUMNS))
plot_distributions(ADDED_COLUMNS, title="Признаки, добавленные в исправленной версии датасета")

,"Доля нулей, %",Минимум,Медиана,Среднее,99-й процентиль,Максимум,"Медиана: Норма, TCP","Медиана: Норма, UDP",Медиана: Атака
Пакетов с флагом RST вперёд,"82,37",0,"0,00","0,28","3,00",190,"0,00","0,00","1,00"
Пакетов с флагом RST обратно,"87,72",0,"0,00","0,14","2,00",12,"0,00","0,00","0,00"
"Общее время TCP-соединения, мкс","47,68",0,"27,00","31 572 341,03","360 180 166,25",30 290 439 669,"10 050 827,00","0,00","42 015,00"


## 8. Чем виды атак отличаются от нормального трафика

Для каждого вида атаки берём медиану признака и сравниваем её с медианой у нормального трафика TCP. Норму берём только TCP, потому что атаки почти целиком идут по этому протоколу. Для сравнения выбраны по одному-два признака из основных пунктов документа.

In [55]:
PROFILE_COLUMNS = [
    "Flow Duration",
    "Total Fwd Packet",
    "Total Bwd packets",
    "Total Length of Fwd Packet",
    "Total Length of Bwd Packet",
    "Packet Length Mean",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "SYN Flag Count",
    "FIN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "FWD Init Win Bytes",
    "Bwd Init Win Bytes",
    "Fwd Act Data Pkts",
    "Idle Mean",
]

normal_tcp_medians = df.loc[is_benign & is_tcp, PROFILE_COLUMNS].median().rename("Норма, TCP")
attack_medians = attacks.groupby("Label")[PROFILE_COLUMNS].median()

# Строки это признаки, столбцы это норма и виды атак
show_table(pd.concat([normal_tcp_medians, attack_medians.T], axis=1), digits=0)

,"Норма, TCP",Botnet,DDoS,DoS GoldenEye,DoS Hulk,DoS Slowhttptest,DoS Slowloris,FTP-Patator,Heartbleed,Infiltration,Infiltration - Portscan,Portscan,SSH-Patator,Web Attack - Brute Force,Web Attack - SQL Injection,Web Attack - XSS
"Длительность потока, мкс",5 958 021,76 684,6 515 832,11 922 141,162 984,108 144 136,102 629 309,8 995 452,119 261 118,100 186 533,48,47,12 122 527,34 092 796,5 010 373,67 745 305
Пакетов вперёд,12,5,8,9,8,4,5,11,2 791,34,2,1,22,204,6,205
Пакетов обратно,10,4,5,5,6,3,3,17,2 070,34,1,1,33,105,5,105
Байт вперёд,717,195,20,424,353,530,247,120,7 920,12 051,0,0,2 008,43 906,599,48 783
Байт обратно,3 597,128,11 595,11 632,11 595,0,0,188,7 883 880,0,0,0,2 745,72 190,2 021,183 599
"Средний размер пакета, байт",189,36,893,763,853,117,82,11,1 619,169,0,0,86,376,242,747
Байт в секунду,295,4 392,1 794,511,73 359,23,24,34,66 172,247,0,0,393,3 401,488,3 418
Пакетов в секунду,2,121,2,1,84,0,0,3,41,1,62 500,42 553,5,9,2,5
"Средний интервал между пакетами, мкс",519 428,9 265,533 822,1 185 576,12 768,10 135 174,6 414 730,333 274,24 494,873 755,23,47,224 333,110 691,556 303,218 563
Пакетов с флагом SYN,2,2,2,2,2,2,2,2,0,0,2,1,2,2,2,2


In [56]:
# Во сколько раз медиана у атаки отличается от медианы у нормы. Считаем в порядках:
# 1 означает "в 10 раз больше", -2 означает "в 100 раз меньше", 0 означает "столько же".
# К медианам прибавляем 1, чтобы не делить на ноль.
difference = np.log10((attack_medians + 1) / (normal_tcp_medians + 1))

# Для подсказки при наведении: сами медианы у атаки и у нормы.
# Строки таблицы подсказок это признаки, столбцы это атаки, как и на графике.
attack_hints = attack_medians.T.values
normal_hints = np.tile(normal_tcp_medians.values.reshape(-1, 1), len(attack_medians))

fig = go.Figure(
    go.Heatmap(
        z=difference.T.values,
        x=difference.index,
        y=[RU[column] for column in PROFILE_COLUMNS],
        customdata=np.dstack([attack_hints, normal_hints]),
        colorscale=BLUE_GRAY_RED,
        zmin=-4,
        zmax=4,
        xgap=2,
        ygap=2,
        colorbar=dict(
            title="Медиана у атаки<br>по сравнению с нормой",
            tickvals=[-4, -2, 0, 2, 4],
            ticktext=["в 10 000 раз меньше", "в 100 раз меньше", "такая же", "в 100 раз больше", "в 10 000 раз больше"],
        ),
        hovertemplate=(
            "%{x}<br>%{y}<br>медиана у атаки: %{customdata[0]:,.0f}"
            "<br>медиана у нормы TCP: %{customdata[1]:,.0f}<extra></extra>"
        ),
    )
)
style(fig, title="Медианы признаков у атак по сравнению с нормальным трафиком TCP", height=760)
fig.update_layout(margin_l=300)
fig.update_yaxes(autorange="reversed")
fig.show()

## 9. Связи между признаками

Считаем корреляцию Спирмена. Она сравнивает не сами значения, а их порядок по возрастанию, поэтому редкие огромные значения её не искажают. Расчёт идёт на случайных 200 тысячах потоков: этого достаточно, а считается намного быстрее.

In [57]:
# Номера, порты и коды не являются измерениями, корреляцию для них не считаем
NOT_MEASUREMENTS = ["id", "Src Port", "Dst Port", "Protocol", "ICMP Code", "ICMP Type", "Attempted Category"]
measurement_columns = [column for column in numeric.columns if column not in NOT_MEASUREMENTS]

sample = df[measurement_columns].sample(200_000, random_state=42)
correlation = sample.corr(method="spearman")

In [58]:
correlation_ru = correlation.rename(index=RU, columns=RU)

fig = correlation_ru.iplot(kind="heatmap", colorscale="rdbu", asFigure=True)
fig.update_traces(
    colorscale=BLUE_GRAY_RED,
    zmin=-1,
    zmax=1,
    colorbar=dict(title="Корреляция"),
    hovertemplate="%{x}<br>%{y}<br>корреляция: %{z:.2f}<extra></extra>",
)
style(
    fig,
    title=(
        "Корреляция Спирмена между числовыми признаками"
        "<br><sup>По горизонтали признаки идут в том же порядке, что и по вертикали. "
        "Названия пары видны при наведении</sup>"
    ),
    height=1150,
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(autorange="reversed", tickfont_size=9, dtick=1)
fig.show()

### Скореллированные признаки

In [59]:
def share_of_equal_values(first, second):
    """Доля потоков, где значения двух колонок равны, в процентах.

    Разницу меньше одной миллионной считаем ошибкой округления, а не различием.
    """
    return np.isclose(df[first], df[second], atol=1e-6).mean() * 100


# Каждая пара встречается в таблице корреляций дважды. Оставляем половину выше диагонали.
above_diagonal = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
pairs = correlation.where(above_diagonal).stack().rename("Корреляция").reset_index()
pairs.columns = ["Колонка 1", "Колонка 2", "Корреляция"]

strong_pairs = pairs[pairs["Корреляция"].abs() >= 0.99].copy()
strong_pairs["Потоков с равными значениями, %"] = [
    share_of_equal_values(first, second)
    for first, second in zip(strong_pairs["Колонка 1"], strong_pairs["Колонка 2"])
]
strong_pairs["Признак 1"] = strong_pairs["Колонка 1"].map(RU)
strong_pairs["Признак 2"] = strong_pairs["Колонка 2"].map(RU)
strong_pairs = strong_pairs.sort_values("Корреляция", ascending=False).reset_index(drop=True)

print("Пар с корреляцией не меньше 0,99:", len(strong_pairs))
show_table(strong_pairs[["Признак 1", "Признак 2", "Корреляция", "Потоков с равными значениями, %"]], digits=3)

Пар с корреляцией не меньше 0,99: 28


,Признак 1,Признак 2,Корреляция,"Потоков с равными значениями, %"
0,Пакетов с флагом URG вперёд,Пакетов с флагом URG,"1,000","100,000"
1,"Средний размер пакета, байт","Средний размер пакета в потоке, байт","1,000","100,000"
2,"Средний размер пакета вперёд, байт","Средний размер сегмента вперёд, байт","1,000","100,000"
3,Ст. отклонение размера пакета,Дисперсия размера пакета,"1,000","16,935"
4,"Средний размер пакета обратно, байт","Средний размер сегмента обратно, байт","1,000","100,000"
5,"Байт в блоке вперёд, в среднем","Пакетов в блоке вперёд, в среднем","1,000","95,860"
6,"Байт в блоке вперёд, в среднем","Скорость передачи блоков вперёд, в среднем","1,000","95,874"
7,"Пакетов в блоке вперёд, в среднем","Скорость передачи блоков вперёд, в среднем","1,000","95,858"
8,"Среднее время простоя, мкс","Наибольшее время простоя, мкс","0,999","89,544"
9,Пакетов в секунду,Пакетов в секунду вперёд,"0,999","2,622"


- Три пары признаков совпадают полностью: 

`Packet Length Mean` и `Average Packet Size` 

`Fwd Packet Length Mean` и `Fwd Segment Size Avg`

`Bwd Packet Length Mean` и `Bwd Segment Size Avg`
- `Fwd URG Flags` и `URG Flag Count` равны почти во всех потоках, потому что оба почти всегда равны нулю.

## 10. Итоги

1. **Объём.** 2 099 976 потоков за пять дней. Нормальный трафик занимает 75,4%, атаки 24,1%, попытки атак 0,6%.
2. **Неравные классы.** Число потоков у атак различается от 159 тысяч (Portscan, DoS Hulk) до 11 (Heartbleed). У пяти видов атак меньше 100 потоков.
3. **Расписание.** Каждый вид атаки записан в один день и в один промежуток времени. В понедельник атак нет.
4. **Попытки атак.** 11 979 потоков помечены как попытки. Перед обучением модели нужно решить, считать их нормой или атакой. Отдельным классом их делать нельзя.
5. **Качество.** Пропусков и повторов нет. Нужно обработать 5 потоков с бесконечной скоростью и 593 потока с отрицательной длиной заголовков. Значение -1 в кодах ICMP и в категории попытки означает «не относится к потоку».
6. **Протоколы.** 63% нормальных потоков идут по UDP, а 99,9% потоков атак по TCP. Сравнивать атаки нужно с нормальным трафиком TCP, иначе разница объясняется протоколом.
7. **Адреса и время.** Большинство атак идёт с адреса `172.16.0.1` на адрес `192.168.10.50` и в известные часы. Адреса, идентификатор потока и время начала описывают стенд и расписание записи, а не поведение потока. Как признаки для модели они не подходят.
8. **Разброс значений.** У большинства числовых признаков значения различаются на несколько порядков, и у многих потоков признак равен нулю. Для моделей, которым важен масштаб признаков, значения стоит логарифмировать.
9. **Лишние признаки.** Три пары признаков совпадают полностью, дисперсия повторяет стандартное отклонение, шесть признаков почти всегда равны нулю. Всего 28 пар признаков имеют корреляцию не меньше 0,99.